# Full Run Research Notebook

This notebook mirrors the current `src/` runtime pipeline as raw in-notebook code.

No imports from `ActionInference.*`, `Autoencoder.*`, or `Sequential.*` are used. The goal is to make the full system inspectable while debugging Runtime Control.

## Research Hypothesis

We want to recover real environment actions from video-only observations.

The full hypothesis is:

$$
\text{video} \rightarrow \Delta z^{desired}
$$

and:

$$
(z_{t-k:t}, a_t) \rightarrow \Delta z^{machine}
$$

Then choose the real action:

$$
 a_t^* = \arg\min_a \left\| \Delta z^{machine}(z_{t-k:t}, a) - \Delta z^{desired} \right\|_2^2
$$

Current finding: Video Learner and MachineForward work, but Runtime Control collapses actions near zero because the two deltas are on different scales.

## Pipeline Equations

Autoencoder:

$$
AE(o_t) = (\hat{o}_t, z_t), \quad z_t \in \mathbb{R}^{64}
$$

Sequential LSTM:

$$
H_t = (z_{t-k+1}, \ldots, z_t)
$$

$$
h_t = LSTM(H_t)
$$

Overlay latent:

$$
M_t = \sum_{i=0}^{k-1} \alpha_i o_{t-k+1+i}
$$

$$
m_t = AE(M_t)_z
$$

Video context:

$$
c_t = [z_t, h_t, m_t]
$$

Latent action and desired delta:

$$
\tilde{a}_t = IDM(c_t)
$$

$$
\Delta z_t^{desired} = FDM(c_t, \tilde{a}_t)
$$

MachineForward:

$$
\Delta z_t^{machine}(a) = MF(z_{t-k:t}, a)
$$

Runtime action search:

$$
a_t^* = \arg\min_a \left(\operatorname{MSE}(\Delta z_t^{machine}(a), s \cdot \Delta z_t^{desired}) + \lambda \|a\|_2^2\right)
$$

where `s = desired_delta_scale` is the next thing to debug.

In [ ]:
from pathlib import Path
import json
import math
import os

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from PIL import Image

try:
    import gymnasium as gym
except Exception as exc:
    gym = None
    print("gymnasium import failed:", exc)

try:
    import imageio.v2 as imageio
except Exception as exc:
    imageio = None
    print("imageio import failed:", exc)


def find_student_root():
    cwd = Path.cwd().resolve()
    for q in [cwd, *cwd.parents]:
        if (q / "src").is_dir() and (q / "runs").is_dir():
            return q
    # Notebook may be under Student/research/full_run.
    for q in [cwd, *cwd.parents]:
        if q.name == "Student":
            return q
    raise RuntimeError("Could not find Student root")

STUDENT = find_student_root()
RUNS = STUDENT / "runs"
ACTION = RUNS / "action_inference"

AE_MODEL_PT = RUNS / "autoencoder" / "model.pth"
SEQ_MODEL_PT = RUNS / "sequential" / "model.pth"
IDM_MODEL_PT = ACTION / "video_learner" / "idm_model.pth"
FDM_MODEL_PT = ACTION / "video_learner" / "fdm_model.pth"
MACHINE_MODEL_PT = ACTION / "machine_forward" / "machine_forward_model.pth"
RUNTIME_STATS_PT = ACTION / "runtime_control" / "runtime_stats.pt"
RUNTIME_ROLLOUT_MP4 = ACTION / "runtime_control" / "runtime_lander_rollout.mp4"

DEVICE = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
print("STUDENT", STUDENT)
print("DEVICE", DEVICE)

## Raw Model Definitions

These classes are copied from the current `src/` architecture so the notebook can be read without jumping through imports.

In [ ]:
class AutoEncoder(nn.Module):
    """84x84 grayscale conv autoencoder."""

    def __init__(self, z_dim: int):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 64, 4, 2, 1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 128, 4, 2, 1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Conv2d(128, 256, 3, 1, 1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.Conv2d(256, 256, 3, 1, 1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(25600, z_dim),
            nn.LayerNorm(z_dim),
        )
        self.decoder_fc = nn.Linear(z_dim, 25600)
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(256, 128, 4, 2, 1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.ConvTranspose2d(128, 64, 4, 2, 1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.ConvTranspose2d(64, 32, 4, 2, 1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.ConvTranspose2d(32, 16, 4, 2, 1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.Conv2d(16, 1, kernel_size=3, padding=1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        z = self.encoder(x)
        x_hat = self.decoder_fc(z).view(-1, 256, 10, 10)
        x_hat = self.decoder(x_hat)
        x_hat = x_hat[:, :, :84, :84]
        return x_hat, z

    def decode(self, z):
        x_hat = self.decoder_fc(z).view(-1, 256, 10, 10)
        x_hat = self.decoder(x_hat)
        return x_hat[:, :, :84, :84]


class LatentLSTM(nn.Module):
    def __init__(self, z_dim: int, hidden_dim: int = 256, num_layers: int = 2, dropout: float = 0.1):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=z_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.head = nn.Sequential(
            nn.LayerNorm(hidden_dim),
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, z_dim),
        )

    def forward(self, z_seq, lengths=None):
        if lengths is not None:
            packed = nn.utils.rnn.pack_padded_sequence(
                z_seq,
                lengths.cpu(),
                batch_first=True,
                enforce_sorted=False,
            )
            _, (h_n, _) = self.lstm(packed)
        else:
            _, (h_n, _) = self.lstm(z_seq)
        return self.head(h_n[-1])


class IDMModel(nn.Module):
    def __init__(self, c_dim, latent_a_dim=8):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(c_dim, 512),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Linear(512, 512),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Linear(512, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Linear(256, 128),
            nn.LayerNorm(128),
            nn.GELU(),
            nn.Linear(128, latent_a_dim),
            nn.Tanh(),
        )

    def forward(self, c_t):
        return self.net(c_t)


class FDMModel(nn.Module):
    def __init__(self, c_dim, z_dim=64, latent_a_dim=8):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(c_dim + latent_a_dim, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Linear(256, 512),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Linear(512, 512),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Linear(512, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Linear(256, 128),
            nn.LayerNorm(128),
            nn.GELU(),
            nn.Linear(128, z_dim),
        )

    def forward(self, c_t, latent_a):
        return self.net(torch.cat([c_t, latent_a], dim=-1))


class MachineForwardModel(nn.Module):
    def __init__(self, z_dim=64, real_a_dim=2, hidden_dim=256, num_layers=1):
        super().__init__()
        self.history_encoder = nn.LSTM(
            input_size=z_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
        )
        self.head = nn.Sequential(
            nn.LayerNorm(hidden_dim + real_a_dim),
            nn.Linear(hidden_dim + real_a_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, z_dim),
        )

    def forward(self, z_hist, a_real):
        _, (h_n, _) = self.history_encoder(z_hist)
        h_t = h_n[-1]
        return self.head(torch.cat([h_t, a_real], dim=-1))

## Runtime Helpers

These functions mirror `runtime_control/control.py`, with one extra explicit parameter:

$$
\Delta z^{desired}_{scaled} = s \cdot \Delta z^{desired}
$$

where `s = desired_delta_scale`.

In [ ]:
def preprocess_rgb(frame):
    img = Image.fromarray(frame).convert("L").resize((84, 84))
    x = np.array(img, dtype=np.float32) / 255.0
    return torch.from_numpy(x)[None, None]


def overlay_frames(x_seq, decay=0.9):
    # x_seq: (B,T,1,84,84)
    T = x_seq.shape[1]
    weights = torch.tensor([decay ** (T - 1 - i) for i in range(T)], device=x_seq.device, dtype=x_seq.dtype)
    weights = weights / weights.sum()
    return (x_seq * weights.view(1, T, 1, 1, 1)).sum(dim=1)


def lstm_predict_and_hidden(lstm_model, z_input):
    h_seq, _ = lstm_model.lstm(z_input)
    h_t = h_seq[:, -1, :]
    z_lstm_next = lstm_model(z_input)
    if isinstance(z_lstm_next, tuple):
        z_lstm_next = z_lstm_next[-1]
    return h_t, z_lstm_next


def pad_tail(items, length):
    if len(items) < length:
        return [items[0]] * (length - len(items)) + items
    return items[-length:]


def tensor_stats(x):
    x = x.detach().float().cpu()
    return {
        "shape": tuple(x.shape),
        "mean": float(x.mean()),
        "std": float(x.std(unbiased=False)),
        "min": float(x.min()),
        "max": float(x.max()),
        "norm_mean": float(torch.linalg.vector_norm(x.reshape(x.shape[0], -1), dim=-1).mean()) if x.ndim > 1 else float(torch.linalg.vector_norm(x)),
        "nan_count": int(torch.isnan(x).sum()),
        "inf_count": int(torch.isinf(x).sum()),
    }

In [ ]:
def load_models(device=DEVICE, latent_a_dim=8):
    ae = AutoEncoder(z_dim=64).to(device)
    ae.load_state_dict(torch.load(AE_MODEL_PT, map_location=device))

    lstm = LatentLSTM(z_dim=64, hidden_dim=256, num_layers=2, dropout=0.1).to(device)
    lstm.load_state_dict(torch.load(SEQ_MODEL_PT, map_location=device))

    idm = IDMModel(c_dim=384, latent_a_dim=latent_a_dim).to(device)
    idm.load_state_dict(torch.load(IDM_MODEL_PT, map_location=device))

    fdm = FDMModel(c_dim=384, z_dim=64, latent_a_dim=latent_a_dim).to(device)
    fdm.load_state_dict(torch.load(FDM_MODEL_PT, map_location=device))

    mf = MachineForwardModel(z_dim=64, real_a_dim=2, hidden_dim=256, num_layers=1).to(device)
    mf.load_state_dict(torch.load(MACHINE_MODEL_PT, map_location=device))

    for model in [ae, lstm, idm, fdm, mf]:
        model.eval()

    return ae, lstm, idm, fdm, mf

models = load_models()
ae_model, lstm_model, idm_model, fdm_model, machine_forward_model = models
print("loaded checkpoints")

## Desired Delta From Video Context

The Video Learner side computes:

$$
c_t = [z_t, h_t, m_t]
$$

$$
\tilde{a}_t = IDM(c_t)
$$

$$
\Delta z^{desired} = FDM(c_t, \tilde{a}_t)
$$

For runtime search we test:

$$
\Delta z^{desired}_{scaled} = s \cdot \Delta z^{desired}
$$

In [ ]:
def desired_delta_from_video_context_raw(
    ae_model,
    lstm_model,
    idm_model,
    fdm_model,
    x_hist,
    history_len=31,
    overlay_decay=0.9,
    delta_z_scale=1.5,
    desired_delta_scale=1.0,
):
    hist = pad_tail(x_hist, history_len)
    x_seq = torch.stack(hist, dim=0)[None]

    x_overlay = overlay_frames(x_seq, decay=overlay_decay)
    _, m_t = ae_model(x_overlay)

    z_hist = torch.stack([ae_model(x_seq[:, t])[1] for t in range(x_seq.shape[1])], dim=1)
    z_t = z_hist[:, -1, :]
    h_t, z_lstm_next = lstm_predict_and_hidden(lstm_model, z_hist)
    c_t = torch.cat([z_t, h_t, m_t], dim=-1)

    latent_a = idm_model(c_t)
    delta_z_raw = fdm_model(c_t, latent_a)
    delta_z_desired = delta_z_raw * delta_z_scale * desired_delta_scale

    debug = {
        "x_overlay": x_overlay,
        "z_hist": z_hist,
        "z_t": z_t,
        "h_t": h_t,
        "m_t": m_t,
        "c_t": c_t,
        "latent_a": latent_a,
        "delta_z_raw": delta_z_raw,
        "delta_z_desired": delta_z_desired,
        "z_lstm_next": z_lstm_next,
    }
    return delta_z_desired, latent_a, debug

## MachineForward Action Search

Sample candidate real actions:

$$
a^{(j)} \sim U([a_{low}, a_{high}]^2)
$$

Predict candidate deltas:

$$
\Delta z_j^{machine} = MF(z_{t-k:t}, a^{(j)})
$$

Choose:

$$
j^* = \arg\min_j \left( \operatorname{MSE}(\Delta z_j^{machine}, \Delta z^{desired}_{scaled}) + \lambda \|a^{(j)}\|_2^2 \right)
$$

In [ ]:
def choose_action_by_machine_forward_raw(
    machine_forward_model,
    z_hist,
    delta_z_desired,
    n_candidates=8192,
    action_low=-0.7,
    action_high=0.7,
    action_l2_penalty=0.05,
    real_a_dim=2,
    return_debug=True,
):
    bsz, hist_len, z_dim = z_hist.shape
    device = z_hist.device
    dtype = z_hist.dtype

    actions = torch.empty(bsz, n_candidates, real_a_dim, device=device, dtype=dtype).uniform_(action_low, action_high)

    z_hist_rep = z_hist[:, None, :, :].expand(bsz, n_candidates, hist_len, z_dim)
    dz_rep = delta_z_desired[:, None, :].expand(bsz, n_candidates, delta_z_desired.shape[-1])

    z_hist_flat = z_hist_rep.reshape(bsz * n_candidates, hist_len, z_dim)
    a_flat = actions.reshape(bsz * n_candidates, real_a_dim)

    with torch.no_grad():
        dz_pred = machine_forward_model(z_hist_flat, a_flat).reshape(bsz, n_candidates, -1)
        latent_loss = ((dz_pred - dz_rep) ** 2).mean(dim=-1)
        action_penalty = (actions ** 2).mean(dim=-1)
        losses = latent_loss + action_l2_penalty * action_penalty
        best_idx = losses.argmin(dim=1)

    best_actions = actions[torch.arange(bsz, device=device), best_idx]
    best_losses = losses[torch.arange(bsz, device=device), best_idx]

    if not return_debug:
        return best_actions, best_losses

    debug = {
        "actions": actions,
        "dz_pred": dz_pred,
        "latent_loss": latent_loss,
        "action_penalty": action_penalty,
        "losses": losses,
        "best_idx": best_idx,
        "best_actions": best_actions,
        "best_losses": best_losses,
    }
    return best_actions, best_losses, debug

## One-Step Runtime Probe

This probe isolates the current failure:

1. Render current frame.
2. Build video history and machine history.
3. Compute desired delta.
4. Sample candidate real actions.
5. Compare desired delta scale against reachable machine candidate delta scale.

If:

$$
\|\Delta z^{desired}\| \gg \|\Delta z^{machine}\|
$$

then the action search cannot find a useful real action.

In [ ]:
def one_step_runtime_probe(
    desired_delta_scale=1.0,
    delta_z_scale=1.5,
    video_history_len=31,
    machine_hist_len=8,
    overlay_decay=0.9,
    n_action_candidates=8192,
    action_low=-0.7,
    action_high=0.7,
    action_l2_penalty=0.05,
    seed=42,
):
    if gym is None:
        raise RuntimeError("gymnasium is unavailable")
    torch.manual_seed(seed)
    np.random.seed(seed)

    env = gym.make("LunarLanderContinuous-v3", render_mode="rgb_array")
    env.reset(seed=seed)

    x_hist = []
    z_buffer = []

    # Warm up histories with repeated current observation, matching pad_tail behavior.
    rgb = env.render()
    x = preprocess_rgb(rgb).to(DEVICE)
    x_hist.append(x.squeeze(0))
    with torch.no_grad():
        _, z = ae_model(x)
    z_buffer.append(z.squeeze(0).detach().cpu())

    with torch.no_grad():
        delta_desired, latent_a, video_debug = desired_delta_from_video_context_raw(
            ae_model,
            lstm_model,
            idm_model,
            fdm_model,
            x_hist=x_hist,
            history_len=video_history_len,
            overlay_decay=overlay_decay,
            delta_z_scale=delta_z_scale,
            desired_delta_scale=desired_delta_scale,
        )

        machine_hist = pad_tail(z_buffer, machine_hist_len)
        z_hist_machine = torch.stack(machine_hist, dim=0)[None].to(DEVICE)

        action, search_loss, search_debug = choose_action_by_machine_forward_raw(
            machine_forward_model,
            z_hist=z_hist_machine,
            delta_z_desired=delta_desired,
            n_candidates=n_action_candidates,
            action_low=action_low,
            action_high=action_high,
            action_l2_penalty=action_l2_penalty,
            real_a_dim=2,
            return_debug=True,
        )

    env.close()

    candidate_norms = torch.linalg.vector_norm(search_debug["dz_pred"][0], dim=-1)
    result = {
        "desired_delta_scale": desired_delta_scale,
        "delta_z_scale": delta_z_scale,
        "desired_norm": float(torch.linalg.vector_norm(delta_desired).item()),
        "raw_desired_norm": float(torch.linalg.vector_norm(video_debug["delta_z_raw"]).item()),
        "candidate_norm_mean": float(candidate_norms.mean().item()),
        "candidate_norm_max": float(candidate_norms.max().item()),
        "candidate_norm_std": float(candidate_norms.std(unbiased=False).item()),
        "best_action": action[0].detach().cpu().numpy(),
        "best_action_norm": float(torch.linalg.vector_norm(action[0]).item()),
        "best_search_loss": float(search_loss.item()),
        "latent_a_stats": tensor_stats(latent_a),
        "desired_stats": tensor_stats(delta_desired),
        "candidate_delta_stats": tensor_stats(search_debug["dz_pred"][0]),
    }
    return result, video_debug, search_debug

probe_result, video_debug, search_debug = one_step_runtime_probe(desired_delta_scale=1.0)
probe_result

## Desired Delta Scale Sweep

Experiment 002 used adaptive stride:

$$
\Delta z = z_{t+s} - z_t
$$

with:

$$
\mathbb{E}[s] \approx 1.912
$$

A conservative first compensation is:

$$
s = \frac{1}{1.912} \approx 0.52
$$

But TensorBoard suggested a larger mismatch, so we sweep:

$$
s \in \{1.0, 0.52, 0.25, 0.1, 0.05\}
$$

In [ ]:
scales = [1.0, 1 / 1.912294, 0.25, 0.1, 0.05]
rows = []
for s in scales:
    r, _, _ = one_step_runtime_probe(desired_delta_scale=s, seed=42)
    rows.append(r)

for r in rows:
    print(
        f"scale={r['desired_delta_scale']:.4f} "
        f"desired_norm={r['desired_norm']:.4f} "
        f"candidate_mean={r['candidate_norm_mean']:.4f} "
        f"candidate_max={r['candidate_norm_max']:.4f} "
        f"best_action_norm={r['best_action_norm']:.4f} "
        f"loss={r['best_search_loss']:.6f} "
        f"best_action={r['best_action']}"
    )

In [ ]:
scale_values = [r["desired_delta_scale"] for r in rows]
desired_norms = [r["desired_norm"] for r in rows]
cand_means = [r["candidate_norm_mean"] for r in rows]
cand_maxs = [r["candidate_norm_max"] for r in rows]
action_norms = [r["best_action_norm"] for r in rows]

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))

ax[0].plot(scale_values, desired_norms, "o-", label="desired norm")
ax[0].plot(scale_values, cand_means, "o-", label="candidate norm mean")
ax[0].plot(scale_values, cand_maxs, "o-", label="candidate norm max")
ax[0].set_xscale("log")
ax[0].invert_xaxis()
ax[0].set_xlabel("desired_delta_scale")
ax[0].set_ylabel("latent delta norm")
ax[0].set_title("Desired vs reachable MachineForward delta scale")
ax[0].grid(True, alpha=0.3)
ax[0].legend()

ax[1].plot(scale_values, action_norms, "o-", color="tab:red")
ax[1].set_xscale("log")
ax[1].invert_xaxis()
ax[1].set_xlabel("desired_delta_scale")
ax[1].set_ylabel("selected action norm")
ax[1].set_title("Does search stop choosing near-zero actions?")
ax[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Full Rollout Function

This is the raw notebook version of `rollout_lander_machine_forward`, with explicit `desired_delta_scale`.

In [ ]:
def rollout_lander_machine_forward_raw(
    max_steps=500,
    video_history_len=31,
    machine_hist_len=8,
    overlay_decay=0.9,
    n_action_candidates=8192,
    action_low=-0.7,
    action_high=0.7,
    action_l2_penalty=0.05,
    delta_z_scale=1.5,
    desired_delta_scale=1.0,
    seed=42,
    save_mp4=False,
    out_mp4=RUNTIME_ROLLOUT_MP4,
):
    if gym is None:
        raise RuntimeError("gymnasium is unavailable")
    torch.manual_seed(seed)
    np.random.seed(seed)

    env = gym.make("LunarLanderContinuous-v3", render_mode="rgb_array")
    env.reset(seed=seed)

    x_hist = []
    z_buffer = []
    frames = []
    actions = []
    latent_actions = []
    desired_delta_norms = []
    candidate_delta_mean_norms = []
    candidate_delta_max_norms = []
    search_losses = []
    rewards = []
    total_reward = 0.0

    for _ in range(max_steps):
        rgb = env.render()
        frames.append(rgb)
        x = preprocess_rgb(rgb).to(DEVICE)
        x_hist.append(x.squeeze(0))

        with torch.no_grad():
            _, z = ae_model(x)
        z_buffer.append(z.squeeze(0).detach().cpu())

        with torch.no_grad():
            delta_z_desired, latent_a, _ = desired_delta_from_video_context_raw(
                ae_model,
                lstm_model,
                idm_model,
                fdm_model,
                x_hist=x_hist,
                history_len=video_history_len,
                overlay_decay=overlay_decay,
                delta_z_scale=delta_z_scale,
                desired_delta_scale=desired_delta_scale,
            )

            machine_hist = pad_tail(z_buffer, machine_hist_len)
            z_hist_machine = torch.stack(machine_hist, dim=0)[None].to(DEVICE)

            action, search_loss, search_debug = choose_action_by_machine_forward_raw(
                machine_forward_model,
                z_hist=z_hist_machine,
                delta_z_desired=delta_z_desired,
                n_candidates=n_action_candidates,
                action_low=action_low,
                action_high=action_high,
                action_l2_penalty=action_l2_penalty,
                real_a_dim=2,
                return_debug=True,
            )

        cand_norms = torch.linalg.vector_norm(search_debug["dz_pred"][0], dim=-1)
        action_np = action[0].detach().cpu().numpy()
        action_np = np.clip(action_np, -1.0, 1.0).astype(np.float32)

        _, reward, terminated, truncated, _ = env.step(action_np)

        actions.append(action_np.copy())
        latent_actions.append(latent_a[0].detach().cpu().numpy().copy())
        desired_delta_norms.append(float(torch.linalg.vector_norm(delta_z_desired).item()))
        candidate_delta_mean_norms.append(float(cand_norms.mean().item()))
        candidate_delta_max_norms.append(float(cand_norms.max().item()))
        search_losses.append(float(search_loss.item()))
        rewards.append(float(reward))
        total_reward += float(reward)

        if len(x_hist) > video_history_len:
            x_hist = x_hist[-video_history_len:]
        if len(z_buffer) > machine_hist_len:
            z_buffer = z_buffer[-machine_hist_len:]

        if terminated or truncated:
            break

    env.close()

    stats = {
        "frames": frames,
        "total_reward": total_reward,
        "actions": np.array(actions),
        "latent_actions": np.array(latent_actions),
        "desired_delta_norms": np.array(desired_delta_norms),
        "candidate_delta_mean_norms": np.array(candidate_delta_mean_norms),
        "candidate_delta_max_norms": np.array(candidate_delta_max_norms),
        "search_losses": np.array(search_losses),
        "rewards": np.array(rewards),
    }

    if save_mp4 and imageio is not None:
        imageio.mimsave(out_mp4, frames, fps=25)
        print("saved", out_mp4)

    return stats

## Optional Full Runtime Test

Run this after the one-step sweep identifies a promising `desired_delta_scale`.

Start with:

$$
s = 1 / 1.912294 \approx 0.52
$$

Then try smaller values if actions still collapse.

In [ ]:
# Example:
# stats = rollout_lander_machine_forward_raw(desired_delta_scale=1/1.912294, max_steps=500, save_mp4=True)
# print("reward", stats["total_reward"])
# print("action mean", stats["actions"].mean(axis=0))
# print("action std", stats["actions"].std(axis=0))
# print("desired norm mean", stats["desired_delta_norms"].mean())
# print("candidate mean norm", stats["candidate_delta_mean_norms"].mean())
# print("candidate max norm", stats["candidate_delta_max_norms"].mean())

In [ ]:
def plot_runtime_stats(stats):
    actions = stats["actions"]
    steps = np.arange(len(actions))
    fig, ax = plt.subplots(4, 1, figsize=(14, 12), sharex=True)

    ax[0].plot(steps, actions[:, 0], label="a0")
    ax[0].plot(steps, actions[:, 1], label="a1")
    ax[0].axhline(0, color="black", lw=1)
    ax[0].set_ylim(-1.05, 1.05)
    ax[0].set_title("Selected real actions")
    ax[0].grid(True, alpha=0.3)
    ax[0].legend()

    ax[1].plot(steps, stats["desired_delta_norms"], label="desired")
    ax[1].plot(steps, stats["candidate_delta_mean_norms"], label="candidate mean")
    ax[1].plot(steps, stats["candidate_delta_max_norms"], label="candidate max")
    ax[1].set_title("Desired vs reachable candidate delta norms")
    ax[1].grid(True, alpha=0.3)
    ax[1].legend()

    ax[2].plot(steps, stats["search_losses"], color="tab:red")
    ax[2].set_title("Search loss")
    ax[2].grid(True, alpha=0.3)

    ax[3].plot(steps, stats["rewards"], color="tab:green")
    ax[3].set_title("Reward per step")
    ax[3].grid(True, alpha=0.3)
    ax[3].set_xlabel("step")

    plt.tight_layout()
    plt.show()

# Example:
# plot_runtime_stats(stats)

## Current Decision

The first two model stages are considered healthy:

1. Video Learner produces live latent actions and desired deltas.
2. MachineForward learns stronger adaptive-stride deltas and beats zero baseline.

Current bottleneck:

$$
\|\Delta z^{desired}\| \gg \|\Delta z^{machine}(a)\|
$$

so runtime action search collapses to near-zero actions.

Next experiment: introduce and sweep `desired_delta_scale` in Runtime Control.